In [1]:
import numpy as np

dataset_path = "./archive/earlyAgg/"
continuous = np.load(dataset_path + "earlyAgg_continuous.npy")
discrete = np.load(dataset_path + "earlyAgg_discrete.npy")
mortality = np.load(dataset_path + "earlyAgg_mortality_label.npy")
in_mortality_label = np.load(dataset_path + "earlyAgg_in_mortality_label.npy")
arf_label = np.load(dataset_path + "earlyAgg_arf_label.npy")
in_hadm_arf_label = np.load(dataset_path + "earlyAgg_in_arf_label.npy")

In [14]:
from sklearn.model_selection import train_test_split

# stratify_labels = np.array(
#     [str(m) + str(a) + str(ia) for m, a, ia in zip(in_mortality_label, arf_label, in_hadm_arf_label)])

(cont_train, cont_test,
 disc_train, disc_test,
 mortality_train, mortality_test,
 in_mortality_train, in_mortality_test,
 arf_train, arf_test,
 in_arf_train, in_arf_test) = train_test_split(
    continuous, discrete, mortality, in_mortality_label, arf_label, in_hadm_arf_label,
    test_size=0.2,
    random_state=42,
    # stratify=stratify_labels
    stratify=in_mortality_label
)

print("Train shapes:", cont_train.shape, disc_train.shape, mortality_train.shape, in_mortality_train.shape, arf_train.shape,
      in_arf_train.shape)
print("Test shapes :", cont_test.shape, disc_test.shape, mortality_test.shape, in_mortality_test.shape, arf_test.shape, in_arf_test.shape)

# print ratio of label in train and test set
print("Train mortality ratio:", np.mean(mortality_train))
print("Test mortality ratio :", np.mean(mortality_test))
print("Train in_mortality ratio:", np.mean(in_mortality_train))
print("Test in_mortality ratio :", np.mean(in_mortality_test))
print("Train arf ratio      :", np.mean(arf_train))
print("Test arf ratio       :", np.mean(arf_test))
print("Train in_arf ratio   :", np.mean(in_arf_train))
print("Test in_arf ratio    :", np.mean(in_arf_test))

Train shapes: (30809, 217) (30809, 60) (30809,) (30809,) (30809,) (30809,)
Test shapes : (7703, 217) (7703, 60) (7703,) (7703,) (7703,) (7703,)
Train mortality ratio: 0.11392774838521211
Test mortality ratio : 0.11398156562378293
Train in_mortality ratio: 0.11392774838521211
Test in_mortality ratio : 0.11398156562378293
Train arf ratio      : 0.19851342140283684
Test arf ratio       : 0.1934311307282877
Train in_arf ratio   : 0.1618358271933526
Test in_arf ratio    : 0.1608464234713748


In [15]:
np.save(dataset_path + "dataset/train_continuous.npy", cont_train)
np.save(dataset_path + "dataset/train_discrete.npy", disc_train)
np.save(dataset_path + "dataset/train_mortality_label.npy", mortality_train)
np.save(dataset_path + "dataset/train_in_mortality_label.npy", in_mortality_train)
np.save(dataset_path + "dataset/train_arf_label.npy", arf_train)
np.save(dataset_path + "dataset/train_in_arf_label.npy", in_arf_train)
np.save(dataset_path + "dataset/test_continuous.npy", cont_test)
np.save(dataset_path + "dataset/test_discrete.npy", disc_test)
np.save(dataset_path + "dataset/test_mortality_label.npy", mortality_test)
np.save(dataset_path + "dataset/test_in_mortality_label.npy", in_mortality_test)
np.save(dataset_path + "dataset/test_arf_label.npy", arf_test)
np.save(dataset_path + "dataset/test_in_arf_label.npy", in_arf_test)

In [6]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, classification_report
)

# -------------------------
# 1. 데이터 로드
# -------------------------
dataset_path = "./archive/earlyAgg/"
X_train_cont = np.load(dataset_path + "dataset/train_continuous.npy")
X_train_disc = np.load(dataset_path + "dataset/train_discrete.npy")
# y_train = np.load(dataset_path + "dataset/train_mortality_label.npy")
y_train = np.load(dataset_path + "dataset/train_in_mortality_label.npy")
# y_train = np.load(dataset_path + "dataset/train_arf_label.npy")
# y_train = np.load(dataset_path + "dataset/train_in_arf_label.npy")

X_test_cont = np.load(dataset_path + "dataset/test_continuous.npy")
X_test_disc = np.load(dataset_path + "dataset/test_discrete.npy")
# y_test = np.load(dataset_path + "dataset/test_mortality_label.npy")
y_test = np.load(dataset_path + "dataset/test_in_mortality_label.npy")
# y_test = np.load(dataset_path + "dataset/test_arf_label.npy")
# y_test = np.load(dataset_path + "dataset/test_in_arf_label.npy")

# 연속형 + 범주형 데이터 합치기
X_train = np.concatenate([X_train_cont, X_train_disc], axis=1)
# X_train = np.concatenate([X_train_disc], axis=1)
X_test = np.concatenate([X_test_cont, X_test_disc], axis=1)
# X_test = np.concatenate([X_test_disc], axis=1)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

# -------------------------
# 2. XGBoost 모델 정의 및 학습
# -------------------------
clf = XGBClassifier(
    n_estimators=300,  # 트리 개수
    max_depth=6,  # 트리 깊이
    learning_rate=0.05,  # 학습률
    subsample=0.8,  # 데이터 샘플 비율
    colsample_bytree=0.8,  # 피처 샘플 비율
    random_state=2025,
    eval_metric="logloss"  # 경고 방지용
)

clf.fit(X_train, y_train)

# -------------------------
# 3. 예측 및 평가
# -------------------------
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("AUROC    :", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
# precision


Train shape: (30809, 277)  Test shape: (7703, 277)


/home/jgpark/anaconda3/envs/venvpy312/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:50:38] WARNING: /croot/xgboost-split_1749630910898/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy : 0.9071790211605868
Precision: 0.7257617728531855
Recall   : 0.2984054669703872
F1 Score : 0.42292171105730425
AUROC    : 0.8989232104266273

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.99      0.95      6825
           1       0.73      0.30      0.42       878

    accuracy                           0.91      7703
   macro avg       0.82      0.64      0.69      7703
weighted avg       0.89      0.91      0.89      7703



In [20]:
import numpy as np

dataset_path = "./archive/earlyAgg/"
old_label = np.load(dataset_path + "real_earlyAgg_IN_HOSPITAL_MORTALITY_label.npy")
new_label = np.load(dataset_path + "earlyAgg_mortality_label.npy")

In [23]:
(old_label == new_label).all()
import pandas as pd

pd.Series(old_label).value_counts(), pd.Series(new_label).value_counts()

(0    32777
 1     5735
 Name: count, dtype: int64,
 0    34124
 1     4388
 Name: count, dtype: int64)

In [26]:
pd.Series(old_label)

0        0
1        0
2        0
3        0
4        0
        ..
38507    0
38508    0
38509    0
38510    0
38511    0
Length: 38512, dtype: int64

In [27]:
pd.Series(new_label)

0        0
1        0
2        0
3        1
4        0
        ..
38507    0
38508    0
38509    0
38510    0
38511    0
Length: 38512, dtype: int64

In [29]:
pid = np.load("../Synthetic_EHR_Generation/1_real_data/openmimic_preprocessing/earlyAgg/earlyAgg_pids.npy", allow_pickle=True)
pid

array([[     3, 211552],
       [     4, 294638],
       [     6, 228232],
       ...,
       [ 99992, 242052],
       [ 99995, 229633],
       [ 99999, 246512]])